# Phase 3a — Spark analysis

Reads the reviews ingested into HDFS in phase 1 and explores how playtime
relates to whether a player recommends a game, and how that relationship
changes with the game's price.

Requires HDFS to be running (`start-dfs.sh`, `start-yarn.sh`).

In [2]:
import getpass

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [3]:
USER = getpass.getuser()
HDFS_BASE = f"hdfs://localhost:9000/user/{USER}/steam"

INPUT_PATH = f"{HDFS_BASE}/streaming_input/*.csv"
MAPREDUCE_OUTPUT = f"{HDFS_BASE}/output/recommendation_by_playtime/part-*"

spark = (
    SparkSession.builder
    .appName("steam-reviews-analysis")
    .master("local[*]")
    # the default driver memory is tight once the data is cached
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark

26/09/08 20:11:46 WARN Utils: Your hostname, luca-Katana-15-B13VFK resolves to a loopback address: 127.0.1.1; using 192.168.1.18 instead (on interface wlo1)
26/09/08 20:11:46 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/08 20:11:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Loading the data

Each ingested chunk carries its own header, which Spark skips automatically
when `header=True`.

In [4]:
reviews = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(INPUT_PATH)
)

reviews = reviews.cache()

print(f"rows: {reviews.count():,}")
reviews.printSchema()

[Stage 4:================>                                        (5 + 12) / 17]

rows: 500,000
root
 |-- app_id: integer (nullable = true)
 |-- helpful: integer (nullable = true)
 |-- funny: integer (nullable = true)
 |-- date: date (nullable = true)
 |-- is_recommended: boolean (nullable = true)
 |-- hours: double (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- review_id: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- date_release: date (nullable = true)
 |-- win: boolean (nullable = true)
 |-- mac: boolean (nullable = true)
 |-- linux: boolean (nullable = true)
 |-- rating: string (nullable = true)
 |-- positive_ratio: integer (nullable = true)
 |-- user_reviews: integer (nullable = true)
 |-- price_final: double (nullable = true)
 |-- price_original: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- steam_deck: boolean (nullable = true)
 |-- price_bucket: string (nullable = true)



In [5]:
reviews.select(
    "app_id", "title", "hours", "is_recommended", "price_final", "price_bucket", "date"
).show(5, truncate=False)

+------+--------------------------------+-----+--------------+-----------+------------+----------+
|app_id|title                           |hours|is_recommended|price_final|price_bucket|date      |
+------+--------------------------------+-----+--------------+-----------+------------+----------+
|306130|The Elder Scrolls® Online       |558.5|true          |20.0       |mid         |2020-08-19|
|244850|Space Engineers                 |52.7 |true          |20.0       |mid         |2020-08-19|
|730   |Counter-Strike: Global Offensive|823.5|true          |15.0       |mid         |2020-08-19|
|730   |Counter-Strike: Global Offensive|614.7|true          |15.0       |mid         |2020-08-19|
|306130|The Elder Scrolls® Online       |10.0 |false         |20.0       |mid         |2020-08-19|
+------+--------------------------------+-----+--------------+-----------+------------+----------+
only showing top 5 rows



## Playtime buckets

The same thresholds used by the MapReduce mapper, so that the two engines
can be compared directly.

In [6]:
def with_playtime_bucket(df, hours_col="hours", out_col="playtime_bucket"):
    """Add a discrete playtime bucket column, matching the mapper's logic."""
    hours = F.col(hours_col)
    return df.withColumn(
        out_col,
        F.when(hours < 1, "0-1h")
        .when(hours < 5, "1-5h")
        .when(hours < 20, "5-20h")
        .when(hours < 100, "20-100h")
        .otherwise("100h+"),
    )


# Explicit ordering, since the buckets are strings and would otherwise
# sort lexicographically ("100h+" before "20-100h")
BUCKET_ORDER = ["0-1h", "1-5h", "5-20h", "20-100h", "100h+"]

reviews = with_playtime_bucket(reviews)
reviews.groupBy("playtime_bucket").count().show()

+---------------+------+
|playtime_bucket| count|
+---------------+------+
|           0-1h|  6626|
|        20-100h|158660|
|           1-5h| 18484|
|          100h+|250650|
|          5-20h| 65580|
+---------------+------+



## Cross-checking the MapReduce result

The MapReduce job already computed the recommendation rate per playtime
bucket. Recomputing it in Spark and comparing the two outputs verifies that
both engines agree on the same data.

In [7]:
spark_rates = (
    reviews
    .groupBy("playtime_bucket")
    .agg(
        F.count("*").alias("total"),
        F.sum(F.col("is_recommended").cast("int")).alias("recommended"),
    )
    .withColumn("rate", F.round(F.col("recommended") / F.col("total"), 4))
)

spark_rates.orderBy(
    F.expr(f"array_position(array{tuple(BUCKET_ORDER)}, playtime_bucket)")
).show()

[Stage 12:================>                                       (5 + 12) / 17]

+---------------+------+-----------+------+
|playtime_bucket| total|recommended|  rate|
+---------------+------+-----------+------+
|           0-1h|  6626|       2326| 0.351|
|           1-5h| 18484|      10625|0.5748|
|          5-20h| 65580|      53913|0.8221|
|        20-100h|158660|     137418|0.8661|
|          100h+|250650|     218341|0.8711|
+---------------+------+-----------+------+



In [8]:
# The MapReduce output is tab-separated with no header
mr_rates = (
    spark.read
    .option("sep", "\t")
    .option("header", False)
    .csv(MAPREDUCE_OUTPUT)
    .toDF("playtime_bucket", "mr_total", "mr_recommended", "mr_rate")
    .select(
        "playtime_bucket",
        F.col("mr_total").cast("long"),
        F.col("mr_recommended").cast("long"),
        F.col("mr_rate").cast("double"),
    )
)

comparison = (
    spark_rates.join(mr_rates, on="playtime_bucket")
    .withColumn("total_matches", F.col("total") == F.col("mr_total"))
    .withColumn("rate_diff", F.abs(F.col("rate") - F.col("mr_rate")))
)

comparison.select(
    "playtime_bucket", "total", "mr_total", "total_matches", "rate", "mr_rate", "rate_diff"
).show()

mismatches = comparison.filter(~F.col("total_matches") | (F.col("rate_diff") > 0.001)).count()
print("MapReduce and Spark agree" if mismatches == 0 else f"WARNING: {mismatches} mismatching buckets")

+---------------+------+--------+-------------+------+-------+---------+
|playtime_bucket| total|mr_total|total_matches|  rate|mr_rate|rate_diff|
+---------------+------+--------+-------------+------+-------+---------+
|           0-1h|  6626|    6626|         true| 0.351|  0.351|      0.0|
|        20-100h|158660|  158660|         true|0.8661| 0.8661|      0.0|
|           1-5h| 18484|   18484|         true|0.5748| 0.5748|      0.0|
|          100h+|250650|  250650|         true|0.8711| 0.8711|      0.0|
|          5-20h| 65580|   65580|         true|0.8221| 0.8221|      0.0|
+---------------+------+--------+-------------+------+-------+---------+

MapReduce and Spark agree
